# 01 — Generate Precomputed SAM HDF5 Datasets

Run this notebook once on a Colab GPU to generate the three precomputed SAM (`vit_b`) HDF5 files:
- `dataset/SAM_outputs_clean.h5`
- `dataset/SAM_outputs_noise.h5` (in-memory Gaussian noise, `var=0.001`)
- `dataset/SAM_outputs_compression.h5` (in-memory JPEG compression, `quality=90`)

In [ ]:
import os
from pathlib import Path
import subprocess
import sys
import urllib.request

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/facebookresearch/segment-anything.git",
    "opencv-python", "h5py",
])

CHECKPOINT = "sam_vit_b_01ec64.pth"
if not Path(CHECKPOINT).exists():
    urllib.request.urlretrieve(
        f"https://dl.fbaipublicfiles.com/segment_anything/{CHECKPOINT}",
        CHECKPOINT,
    )
print("Checkpoint ready:", CHECKPOINT)

In [ ]:
import torch
from segment_anything import SamAutomaticMaskGenerator, sam_model_registry

if "google.colab" in sys.modules and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

device = "cuda" if torch.cuda.is_available() else "cpu"
sam = sam_model_registry["vit_b"](checkpoint=CHECKPOINT).to(device=device)
mask_generator = SamAutomaticMaskGenerator(sam)
print(f"Loaded SAM vit_b on {device}")

In [ ]:
from coverless_steg import generate_sam_h5

IMAGES_DIR = "/content/drive/MyDrive/coco_dataset/train2017"
OUTPUT_DIR = Path("dataset")
NUM_IMAGES = 500

clean_h5 = generate_sam_h5(
    OUTPUT_DIR / "SAM_outputs_clean.h5",
    IMAGES_DIR,
    mask_generator,
    num_images=NUM_IMAGES,
)
noise_h5 = generate_sam_h5(
    OUTPUT_DIR / "SAM_outputs_noise.h5",
    IMAGES_DIR,
    mask_generator,
    with_noise=True,
    noise_variance=0.001,
    num_images=NUM_IMAGES,
)
comp_h5 = generate_sam_h5(
    OUTPUT_DIR / "SAM_outputs_compression.h5",
    IMAGES_DIR,
    mask_generator,
    with_compression=True,
    jpeg_quality=90,
    num_images=NUM_IMAGES,
)
print("Generated:", clean_h5, noise_h5, comp_h5)

In [ ]:
import h5py

for path in (clean_h5, noise_h5, comp_h5):
    with h5py.File(path, "r") as f:
        keys = sorted(f.keys())
        g0 = f[keys[0]]
        print(f"{path.name}: {len(keys)} groups | 0000 attrs={dict(g0.attrs)} | datasets={list(g0.keys())}")